# Real-Time Amazon Review Scraper
This notebook demonstrates how to scrape customer reviews, ratings, comment tags (titles), and names from an **Amazon** product review page using Python (`requests` and `BeautifulSoup`).

> **Warning:** Amazon has robust anti-scraping mechanisms. If you see a `503 Service Unavailable` error or "No review containers found", it means Amazon has blocked the automated request and served a CAPTCHA page. 
>
> For large-scale or production environments, you would need to use rotating proxies, API services (like ScraperAPI), or browser automation frameworks (like Selenium or Playwright) to bypass these restrictions. This script serves as a foundational example of extracting the right HTML elements (`data-hook` properties) that Amazon uses for its reviews.

In [ ]:
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
def scrape_amazon_reviews(url):
    """
    Scrapes reviews from an Amazon product reviews URL.
    """
    # Headers are critical for Amazon. We mimic a real, modern browser.
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US, en;q=0.5',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none',
        'Sec-Fetch-User': '?1'
    }
    
    print(f"Fetching URL: {url}")
    
    # Use a session to maintain connection state
    session = requests.Session()
    response = session.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to retrieve page: HTTP {response.status_code}")
        print("Amazon might have blocked the request (HTTP 503 usually means a CAPTCHA was triggered).")
        return []
        
    soup = BeautifulSoup(response.content, 'html.parser')
    reviews_data = []
    
    # Amazon uses specific 'data-hook' attributes for its review containers
    review_elements = soup.find_all('div', {'data-hook': 'review'})
    
    if not review_elements:
        print("No review containers found. Amazon likely served a CAPTCHA instead of the product page.")
        return []

    for review in review_elements:
        try:
            # 1. Extract Customer Name
            name_elem = review.find('span', class_='a-profile-name')
            customer_name = name_elem.text.strip() if name_elem else 'N/A'
            
            # 2. Extract Rating (e.g., '4.0 out of 5 stars')
            rating_elem = review.find('i', {'data-hook': 'review-star-rating'})
            rating = rating_elem.text.strip() if rating_elem else 'N/A'
            if rating == 'N/A':
                 # Sometimes Amazon uses a different hook for global reviews
                 rating_elem = review.find('i', {'data-hook': 'cmps-review-star-rating'})
                 rating = rating_elem.text.strip() if rating_elem else 'N/A'
            
            # Clean up rating to just get the number
            rating_val = rating.split(' out of')[0] if 'out of' in rating else rating
            
            # 3. Extract Comment Tag / Title
            tag_elem = review.find('a', {'data-hook': 'review-title'})
            if tag_elem:
                 # Titles usually have spans inside them, the last one being the actual text
                 title_spans = tag_elem.find_all('span')
                 comment_tag = title_spans[-1].text.strip() if title_spans else tag_elem.text.strip()
            else:
                 # Global reviews without an anchor link
                 tag_elem = review.find('span', {'data-hook': 'review-title'})
                 comment_tag = tag_elem.text.strip() if tag_elem else 'N/A'
            
            # 4. Extract Review Comment
            comment_elem = review.find('span', {'data-hook': 'review-body'})
            comment = comment_elem.text.strip() if comment_elem else 'N/A'
            
            reviews_data.append({
                'Customer Name': customer_name,
                'Rating': rating_val,
                'Comment Tag': comment_tag,
                'Review': comment
            })
        except Exception as e:
            print(f"Error parsing a review element: {e}")
            continue
        
    return reviews_data

In [ ]:
# Example Usage:
# This is a live link to an Amazon product's "All Reviews" page (Sony Headphones as an example)
PRODUCT_URL = "https://www.amazon.com/product-reviews/B09S12X76L/ref=cm_cr_dp_d_show_all_btm?ie=UTF8&reviewerType=all_reviews"

print("Initiating scrape...")
scraped_data = scrape_amazon_reviews(PRODUCT_URL)

if scraped_data:
    # Convert the extracted list of dictionaries to a Pandas DataFrame
    df = pd.DataFrame(scraped_data)
    
    print(f"\nSuccessfully scraped {len(df)} reviews!")
    display(df.head())
    
    # Save the scraped data to a CSV file locally
    csv_filename = 'amazon_reviews.csv'
    df.to_csv(csv_filename, index=False)
    print(f"\nData has been saved to '{csv_filename}'")
else:
    print("\nScraping returned no data.")